# Set up library

In [45]:
import numpy as np
import librosa
import librosa.display
import matplotlib.pyplot as plt
from pathlib import Path
import h5py
import os
import librosa
import json
import unicodedata
import pandas as pd






# Helpful function

In [46]:
hop_length = 512
n_bins=168
sr = 48000
bins_per_octave=24
fmin=librosa.note_to_hz('C3')

## get cqt

In [47]:
def get_CQT_of_a_audio(y, sr):

    C = librosa.cqt(
        y,
        sr=sr,
        hop_length=hop_length,
        fmin=fmin,
        n_bins=n_bins,
        bins_per_octave=bins_per_octave
    )

    return C

## Helper to create csv file

In [48]:
def create_label_template(audio_path, output_csv, hop_length=512):

    y, sr = librosa.load(audio_path, sr=48000)

    duration = librosa.get_duration(y=y, sr=sr)

    frame_time = hop_length / sr

    total_frames = int(duration / frame_time)

    frames = np.arange(total_frames)

    start_times = frames * frame_time
    end_times = start_times + frame_time

    df = pd.DataFrame({
        "frame": frames,
        "start_time": start_times,
        "end_time": end_times,
        "label": "Default"
    })

    df.to_csv(output_csv, index=False)

def create_labels_for_audios_in_folder(audio_folder, audio_folder_name = "audio", csv_folder_name = "frame_labels"):


    if os.path.exists(audio_folder):
        for root, dirs, files in os.walk(audio_folder):
            for file in files:
                if file.endswith(".wav"):
                    csv_root = root.replace(audio_folder_name, csv_folder_name)
                    if os.path.exists(csv_root):
                        csv_file_name = file.replace(".wav", ".csv")
                        csv_path = os.path.join(csv_root, csv_file_name)
                        audio_path = os.path.join(root, file)
                        create_label_template(audio_path, csv_path)


                else:
                    print(csv_root, "doesnt exists !")

    else:
        print("Cant find folder at", audio_folder)


## helper to create spectrogrames of audio

In [49]:
def _plot_cqt_spectrogram(C_db, sr, audio_path, img_path=None, is_saving=True, start_frame=0):

    n_frames = C_db.shape[1]
    frame_time = hop_length / sr
    end_frame = start_frame + n_frames - 1

    fig, ax = plt.subplots(figsize=(12, 6))

    img = librosa.display.specshow(
        C_db,
        sr=sr,
        hop_length=hop_length,
        x_axis='time',
        y_axis='cqt_note',
        ax=ax
    )

    frame_positions = np.arange(n_frames) * frame_time
    for t in frame_positions:
        ax.axvline(x=t, color='white', alpha=0.3, linewidth=0.7)

    tick_count = min(10, n_frames)
    if tick_count == 1:
        frame_ticks = np.array([start_frame])
    else:
        frame_ticks = np.linspace(start_frame, end_frame, num=tick_count, dtype=int)
        frame_ticks = np.unique(np.concatenate(([start_frame], frame_ticks, [end_frame])))

    frame_tick_times = (frame_ticks - start_frame) * frame_time
    ax.set_xticks(frame_tick_times)
    ax.set_xticklabels(frame_ticks)
    ax.set_xlabel('Frame index')
    ax.set_title(f'{audio_path} | frames {start_frame} -> {end_frame}')

    fig.colorbar(img, ax=ax, format='%+2.0f dB')
    fig.tight_layout()

    if is_saving and img_path is not None:
        fig.savefig(img_path, dpi=300)

    plt.show()
    plt.close(fig)

def analyze_spectrogram_by_CQT_of_audio(audio_path, img_path, is_saving=True, split_audio=False, segment_duration=1.0):

    if segment_duration <= 0:
        raise ValueError('segment_duration must be greater than 0.')

    y, audio_sr = librosa.load(audio_path, sr=sr)
    C = get_CQT_of_a_audio(y, audio_sr)
    C_db = librosa.amplitude_to_db(np.abs(C))

    if not split_audio:
        _plot_cqt_spectrogram(C_db, audio_sr, audio_path, img_path=img_path, is_saving=is_saving, start_frame=0)
        return

    frame_time = hop_length / audio_sr
    frames_per_segment = max(1, int(np.round(segment_duration / frame_time)))
    img_path_obj = Path(img_path) if img_path is not None else None

    for segment_index, start_frame in enumerate(range(0, C_db.shape[1], frames_per_segment)):
        end_frame_exclusive = min(start_frame + frames_per_segment, C_db.shape[1])
        segment_C_db = C_db[:, start_frame:end_frame_exclusive]

        segment_img_path = None
        if img_path_obj is not None:
            segment_img_path = img_path_obj.with_name(
                f'{img_path_obj.stem}_part_{segment_index:03d}_frames_{start_frame}_{end_frame_exclusive - 1}{img_path_obj.suffix}'
            )

        _plot_cqt_spectrogram(
            segment_C_db,
            audio_sr,
            audio_path,
            img_path=segment_img_path,
            is_saving=is_saving,
            start_frame=start_frame
        )


In [50]:
def analyzing_spectrograms_for_audios_in_folder(audio_folder, split_audio=False, segment_duration=1.0):

    audio_folder_name = "audio"
    img_folder_name = "spectrograms"

    if os.path.exists(audio_folder):
        for root, dirs, files in os.walk(audio_folder):
            for file in files:
                if file.endswith(".wav"):
                    img_root = root.replace(audio_folder_name, img_folder_name)
                    if os.path.exists(img_root):
                        img_file_name = file.replace(".wav", ".png")
                        img_path = os.path.join(img_root, img_file_name)
                        audio_path = os.path.join(root, file)
                        analyze_spectrogram_by_CQT_of_audio(
                            audio_path,
                            img_path,
                            split_audio=split_audio,
                            segment_duration=segment_duration
                        )


                    else:
                        print(img_root, "doesnt exists !")

    else:
        print("Cant find folder at", audio_folder)

# Helper to get quantities notes of audio

In [51]:

save_note_imge_path = f"./Data/notes/qeej2"

def get_notes_of_a_audio(notes, counts, file_name):

    note_midi = [librosa.note_to_midi(n) for n in notes]
    sorted_idx = np.argsort(note_midi)
    notes = notes[sorted_idx]
    counts = counts[sorted_idx]    

    plt.figure(figsize=(14,6))
    plt.bar(notes, counts, width=0.5)

    plt.xlabel("Notes")
    plt.ylabel("Counts")
    plt.title("Histogram of Notes")

    plt.xticks(rotation=45)
    plt.tight_layout()
    save_path = os.path.join(save_note_imge_path, file_name)
    plt.savefig(save_path , dpi=300)
    plt.close()

In [52]:

def extract_topk_pitch_to_json( cqt, sr, file_name, k=7, threshold = 0.1):

    n_frames = cqt.shape[1]
    max_energy = np.max(cqt)

    freqs = librosa.cqt_frequencies(
        n_bins=n_bins,
        fmin=fmin,
        bins_per_octave=bins_per_octave
    )


    times = librosa.frames_to_time(
        np.arange(n_frames),
        sr=sr
    )

    results = []

    notes = []

    for t in range(n_frames):
        frame = cqt[:, t]

        if frame.size == 0:
            continue

        top_k_idx = np.argsort(frame)[-k:][::-1]

        pitches = []
        for i in top_k_idx:
            current_energy = frame[i]
            if float(current_energy) < threshold * max_energy:
                continue

            pitches.append({
                "hz": float(freqs[i]),
                "note": librosa.hz_to_note(freqs[i]),
                "energy": float(frame[i])
            })

            notes.append(librosa.hz_to_note(freqs[i]))
        
        if len(pitches) > 0:
            results.append({
                "frame": int(t),
                "time": float(times[t]),
                "pitches": pitches
            })


    notes, counts = np.unique(notes, return_counts=True)
    note_count_dict = dict(zip(notes, counts))

    get_notes_of_a_audio(notes, counts, file_name.replace(".json", ".png"))

def extract_pitches_from_audios(audio_folder_path):
    if os.path.exists(audio_folder_path):
        for root, dirs, files in os.walk(audio_folder_path):
            for file in files:
                if file.endswith(".wav"):
                    json_file_name = file.replace(".wav",".json")
                    audio_path = os.path.join(root, file)
                    y, sr = librosa.load(audio_path, sr=48000)
                    cqt = get_CQT_of_a_audio(y, sr)
                    extract_topk_pitch_to_json(cqt, sr, json_file_name, threshold=0.1)
                    
    else:
        print("Cant find folder at", audio_folder_path)

## Create Csv file

In [53]:
# audio_folder = "./Data/audio/Full/qeej8"
# create_labels_for_audios_in_folder(audio_folder, csv_folder_name = "label")

In [54]:
# audio_folder = "./Data/audio/Full/qeej8"
# analyzing_spectrograms_for_audios_in_folder(audio_folder)

In [55]:
audio_folder_path = "./Data/audio/Full/qeej2/Đơn_ống"
extract_pitches_from_audios(audio_folder_path)

C:\Users\Admin\AppData\Local\Temp\ipykernel_27380\1071395608.py:33: ComplexWarning: Casting complex values to real discards the imaginary part
  if float(current_energy) < threshold * max_energy:
C:\Users\Admin\AppData\Local\Temp\ipykernel_27380\1071395608.py:39: ComplexWarning: Casting complex values to real discards the imaginary part
  "energy": float(frame[i])
